# Uber Ride Booking Analysis
## Phase 1 – Data Understanding & Initial Audit

## 1. Importing Required Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Loading Dataset (All 5 Tables)

In [3]:
customers = pd.read_csv('Customers.csv')
drivers = pd.read_csv('Drivers.csv')
vehicles = pd.read_csv('Vehicles.csv')
rides = pd.read_csv('Rides.csv')
ride_status = pd.read_csv('Ride_Status.csv')

## 3. Structural Audit of Each Table
In this section, we examine structure, data types, missing values and duplicates.

In [4]:
def audit_table(df, name):
    print(f"TABLE NAME: {name}")

    print(f"\nShape: {df.shape}")
    
    print("\nColumn Data Types:")
    print(df.dtypes)
    
    print("\nMissing Values:")
    print([i for i in df.columns if df[i].isnull().sum()>0])
    
    print("\nDuplicate Rows:", df.duplicated().sum())
    
    print("\nBasic Statistics:")
    print(df.describe())

In [5]:
audit_table(customers,'Customers')

TABLE NAME: Customers

Shape: (13045, 3)

Column Data Types:
Customer_ID             object
Customer_Signup_Date    object
Customer_City           object
dtype: object

Missing Values:
['Customer_City']

Duplicate Rows: 0

Basic Statistics:
       Customer_ID Customer_Signup_Date Customer_City
count        13045                13045         12780
unique       13045                  547            19
top      CUST13045           2024-05-11         Delhi
freq             1                   66          1036


In [6]:
audit_table(drivers,'Drivers')

TABLE NAME: Drivers

Shape: (700, 4)

Column Data Types:
Driver_ID                  object
Driver_Joining_Date        object
Driver_Experience_Years     int64
Driver_Base_City           object
dtype: object

Missing Values:
[]

Duplicate Rows: 0

Basic Statistics:
       Driver_Experience_Years
count               700.000000
mean                  4.972857
std                   2.590973
min                   1.000000
25%                   3.000000
50%                   5.000000
75%                   7.000000
max                   9.000000


In [7]:
audit_table(vehicles,'Vehicles')

TABLE NAME: Vehicles

Shape: (4, 3)

Column Data Types:
Vehicle_Type    object
Base_Fare        int64
Per_KM_Rate      int64
dtype: object

Missing Values:
[]

Duplicate Rows: 0

Basic Statistics:
        Base_Fare  Per_KM_Rate
count    4.000000     4.000000
mean    70.000000    11.250000
std     25.819889     2.986079
min     40.000000     8.000000
25%     55.000000     9.500000
50%     70.000000    11.000000
75%     85.000000    12.750000
max    100.000000    15.000000


In [8]:
audit_table(rides,'Rides')

TABLE NAME: Rides

Shape: (45225, 16)

Column Data Types:
Booking_ID             object
Booking_Timestamp      object
Customer_ID            object
Driver_ID              object
Vehicle_Type           object
Pickup_Location        object
Drop_Location          object
Ride_Distance_KM      float64
Payment_Method         object
Peak_Hour_Flag          int64
Surge_Multiplier      float64
Estimated_Ride_Min      int64
Actual_Ride_Min       float64
Estimated_Fare        float64
Booking_Status         object
Final_Paid_Amount     float64
dtype: object

Missing Values:
['Pickup_Location', 'Drop_Location', 'Ride_Distance_KM', 'Payment_Method', 'Actual_Ride_Min', 'Final_Paid_Amount']

Duplicate Rows: 0

Basic Statistics:
       Ride_Distance_KM  Peak_Hour_Flag  Surge_Multiplier  Estimated_Ride_Min  \
count      43872.000000    45225.000000      45225.000000        45225.000000   
mean          13.541464        0.206523          1.192989           40.619613   
std            6.642478        0.40

In [9]:
audit_table(ride_status,'Ride Status')

TABLE NAME: Ride Status

Shape: (45000, 5)

Column Data Types:
Booking_ID               object
Cancelled_By_Customer     int64
Cancelled_By_Driver       int64
Cancellation_Reason      object
Incomplete_Reason        object
dtype: object

Missing Values:
['Cancellation_Reason', 'Incomplete_Reason']

Duplicate Rows: 0

Basic Statistics:
       Cancelled_By_Customer  Cancelled_By_Driver
count           45000.000000         45000.000000
mean                0.090089             0.060067
std                 0.286312             0.237613
min                 0.000000             0.000000
25%                 0.000000             0.000000
50%                 0.000000             0.000000
75%                 0.000000             0.000000
max                 1.000000             1.000000


## 4. Data Cleaning

### 4.1 Checking Null Values

In [10]:
def isnull(data):
    nv = data.isnull().sum()
    nv = nv[nv>0]
    return nv

In [11]:
print('Rides\n',isnull(rides))
print('\nRide Status\n',isnull(ride_status))
print('\nDrivers',isnull(drivers))
print('\nCustomers',isnull(customers))
print('\nVehicles',isnull(vehicles))

Rides
 Pickup_Location      1353
Drop_Location        1353
Ride_Distance_KM     1353
Payment_Method       1353
Actual_Ride_Min      1353
Final_Paid_Amount    1335
dtype: int64

Ride Status
 Cancellation_Reason    38243
Incomplete_Reason      42745
dtype: int64

Drivers Series([], dtype: int64)

Customers Customer_City    265
dtype: int64

Vehicles Series([], dtype: int64)


## 4.2 Cleaning Rides Table
Handling nulls, duplicates and datatype issues.

In [12]:
rides["Booking_Timestamp"] = pd.to_datetime(rides["Booking_Timestamp"])

In [13]:
print('Duplicates Before Drop',rides.duplicated().sum())
rides = rides.drop_duplicates(subset=['Booking_ID'])
print('Duplicates After Drop',rides.duplicated().sum())

Duplicates Before Drop 0
Duplicates After Drop 0


In [14]:
rides["Pickup_Location"] = rides["Pickup_Location"].fillna("Unknown")
rides["Drop_Location"] = rides["Drop_Location"].fillna("Unknown")

rides["Ride_Distance_KM"] = rides["Ride_Distance_KM"].fillna(rides["Ride_Distance_KM"].median())

rides["Payment_Method"] = rides["Payment_Method"].fillna("Unknown")

rides["Actual_Ride_Min"] = rides["Actual_Ride_Min"].fillna(rides["Estimated_Ride_Min"])

In [15]:
rides.loc[rides["Booking_Status"]=="Cancelled", "Final_Paid_Amount"] = 0

mask_incomplete = (rides["Booking_Status"]=="Incomplete") & (rides["Final_Paid_Amount"].isnull())
rides.loc[mask_incomplete, "Final_Paid_Amount"] = rides["Estimated_Fare"] * 0.5

mask_completed = (rides["Booking_Status"]=="Completed") & (rides["Final_Paid_Amount"].isnull())
rides.loc[mask_completed, "Final_Paid_Amount"] = rides["Estimated_Fare"]

## 4.3 Cleaning Ride_Status Table
Fixing cancellation logic and handling null reasons.

In [16]:
print('Duplicates Before Drop',ride_status.duplicated().sum())
ride_status = ride_status.drop_duplicates(subset=["Booking_ID"])
print('Duplicates After Drop',ride_status.duplicated().sum())

Duplicates Before Drop 0
Duplicates After Drop 0


In [17]:
ride_status.loc[(ride_status["Cancelled_By_Customer"]==1) &(ride_status["Cancelled_By_Driver"]==1),"Cancelled_By_Driver"] = 0

In [18]:
ride_status["Cancellation_Reason"] = ride_status["Cancellation_Reason"].fillna("Not Applicable")
ride_status["Incomplete_Reason"] = ride_status["Incomplete_Reason"].fillna("Not Applicable")

## 4.4 Cleaning Other Tables
Changing data type

In [19]:
customers['Customer_Signup_Date'] = customers['Customer_Signup_Date'].str.split().str[0]
customers['Customer_Signup_Date'] = pd.to_datetime(customers['Customer_Signup_Date'])
customers['Customer_City'] = customers['Customer_City'].fillna('Unknown') 

drivers['Driver_Joining_Date'] = pd.to_datetime(drivers['Driver_Joining_Date'])

## 5. Relationship Validation Between Tables
### 5.1 Checking Row Counts Between Rides and Ride_Status

In [20]:
print("Unique Booking_ID in rides:", rides["Booking_ID"].nunique())
print("Unique Booking_ID in ride_status:", ride_status["Booking_ID"].nunique())

Unique Booking_ID in rides: 45000
Unique Booking_ID in ride_status: 45000


### 5.2 Checking Booking_ID Consistency

In [21]:
set(rides["Booking_ID"]) - set(ride_status["Booking_ID"])

set()

### 5.3 Validating Customer_ID in Rides

In [22]:
set(rides["Customer_ID"]) - set(customers["Customer_ID"])

set()

### 5.4 Validating Driver_ID in Rides

In [23]:
set(rides["Driver_ID"]) - set(drivers["Driver_ID"])

set()

## 6. Feature Engineering
Creating analytical columns.

In [24]:
from datetime import datetime

In [25]:
rides["Ride_Date"] = rides["Booking_Timestamp"].dt.date
rides["Ride_Month"] = rides["Booking_Timestamp"].dt.month
rides["Ride_Day"] = rides["Booking_Timestamp"].dt.day_name()
rides["Ride_Hour"] = rides["Booking_Timestamp"].dt.hour

In [26]:
rides["Operational_Loss"] = np.where(rides["Booking_Status"]=="Incomplete",rides["Estimated_Fare"] - rides["Final_Paid_Amount"],0)
rides["Cancellation_Loss"] = np.where(rides["Booking_Status"]=="Cancelled",rides["Estimated_Fare"],0)

In [27]:
rides["Ride_Length_Category"] = pd.cut(rides["Ride_Distance_KM"],
    bins=[0,5,15,100],labels=["Short","Medium","Long"])

## 7. Business Rule Validation
Validating logical consistency of ride data.

In [28]:
# If Ride Cancelled -> Fare should be 0
canc_error = rides[(rides["Booking_Status"]=="Cancelled") & (rides["Final_Paid_Amount"]!=0)]
print("Cancelled Revenue Errors:", len(canc_error))

Cancelled Revenue Errors: 0


In [29]:
# If Ride is Incomplete -> Fare should be partially paid
incomp_error = rides[(rides['Booking_Status']=='Incomplete') & (rides['Final_Paid_Amount']>=rides['Estimated_Fare'])]
print("Incomplete Revenue Errors:", len(incomp_error))

Incomplete Revenue Errors: 51


In [30]:
mask = (rides["Booking_Status"]=="Incomplete") & (rides["Final_Paid_Amount"] >= rides["Estimated_Fare"])
rides.loc[mask,"Final_Paid_Amount"] = rides["Estimated_Fare"] * 0.5

In [31]:
#Only one cancellation flag active
flag_error = ride_status[(ride_status["Cancelled_By_Customer"]==1) & (ride_status["Cancelled_By_Driver"]==1)]
print("Double Cancellation Errors:", len(flag_error))

Double Cancellation Errors: 0


In [32]:
print("Errors After Fix:",len(rides[(rides["Booking_Status"]=="Cancelled") & (rides["Final_Paid_Amount"]!=0)]))

Errors After Fix: 0


## 8. Merging Rides and Ride_Status for Analysis Preview

In [33]:
df = rides.merge(ride_status, on="Booking_ID", how="left")
print(df.shape)
df.head()

(45000, 27)


,Booking_ID,Booking_Timestamp,Customer_ID,Driver_ID,Vehicle_Type,Pickup_Location,Drop_Location,Ride_Distance_KM,Payment_Method,Peak_Hour_Flag,...,Ride_Month,Ride_Day,Ride_Hour,Operational_Loss,Cancellation_Loss,Ride_Length_Category,Cancelled_By_Customer,Cancelled_By_Driver,Cancellation_Reason,Incomplete_Reason
0,BK000001,2024-03-07 18:07:00,CUST00481,DR00077,Mini,Noida Sector 18,Karol Bagh,20.25,Card,1,...,3,Thursday,18,0.0,0.00,Long,0,0,Not Applicable,Not Applicable
1,BK000002,2024-04-13 20:04:00,CUST09666,DR00400,Sedan,Connaught Place,Lajpat Nagar,15.67,Cash,0,...,4,Saturday,20,0.0,0.00,Long,0,0,Not Applicable,Not Applicable
2,BK000003,2024-04-16 19:32:00,CUST00560,DR00374,SUV,Rajouri Garden,Connaught Place,19.94,UPI,0,...,4,Tuesday,19,0.0,0.00,Long,0,0,Not Applicable,Not Applicable
3,BK000004,2024-06-11 21:16:00,CUST03398,DR00012,Prime,Noida Sector 62,Connaught Place,17.73,Wallet,1,...,6,Tuesday,21,0.0,651.39,Long,1,0,Changed Plan,Not Applicable
4,BK000005,2024-02-10 09:21:00,CUST04092,DR00487,Mini,MG Road Gurgaon,Rajouri Garden,24.14,UPI,0,...,2,Saturday,9,0.0,0.00,Long,0,0,Not Applicable,Not Applicable


## 9. Exploratory Business Insights

### 9.1. Initial Business Metrics Overview

In [34]:
print("Total Revenue:", df["Final_Paid_Amount"].sum())
print("Total Rides:", len(df))
print("Cancellation Rate(%):", (df["Booking_Status"]=="Cancelled").mean()*100)
print("Incomplete Rate (%):", (df["Booking_Status"]=="Incomplete").mean()*100)
df['Delay_Min'] = df['Actual_Ride_Min'] - df['Estimated_Ride_Min']
print("Average Delay (minutes):", df["Delay_Min"].mean())

Total Revenue: 9845822.11
Total Rides: 45000
Cancellation Rate(%): 15.015555555555554
Incomplete Rate (%): 5.011111111111111
Average Delay (minutes): 3.4492


### 9.2 Revenue Distribution by Booking Status

In [35]:
df.groupby('Booking_Status')['Final_Paid_Amount'].sum().reset_index()

,Booking_Status,Final_Paid_Amount
0,Cancelled,0.00
1,Completed,9515994.36
2,Incomplete,329827.75


### 9.3 Overall Revenue Leakage Percentage

In [36]:
print("Operational Leakage %:",(df["Operational_Loss"].sum()*100)/df["Estimated_Fare"].sum())
print("Cancellation Opportunity %:",(df["Cancellation_Loss"].sum()*100)/df["Estimated_Fare"].sum())

print("Total Revenue Impact %:",((df["Operational_Loss"].sum()+df["Cancellation_Loss"].sum())*100)/df["Estimated_Fare"].sum())

Operational Leakage %: 2.1994870405084828
Cancellation Opportunity %: 15.054021319515721
Total Revenue Impact %: 17.253508360024206


### 9.4 Cancellation Rate by Peak vs Non-Peak

In [37]:
df.groupby('Peak_Hour_Flag')['Booking_Status'].value_counts(normalize=True).reset_index()

,Peak_Hour_Flag,Booking_Status,proportion
0,0,Completed,0.800302
1,0,Cancelled,0.150326
2,0,Incomplete,0.049371
3,1,Completed,0.797546
4,1,Cancelled,0.149500
5,1,Incomplete,0.052954


### 9.5 Average Delay Analysis by Vehicle Type

In [38]:
df.groupby('Vehicle_Type')['Delay_Min'].mean().reset_index()

,Vehicle_Type,Delay_Min
0,Mini,3.481247
1,Prime,3.373998
2,SUV,3.456259
3,Sedan,3.483785


### 9.6 Top 10 Revenue Generating Locations

In [39]:
df.groupby('Pickup_Location')['Final_Paid_Amount'].sum().sort_values(ascending=False).head(10).reset_index()

,Pickup_Location,Final_Paid_Amount
0,Rajouri Garden,658229.940
1,Indirapuram,656634.475
2,Saket,650395.240
3,Karol Bagh,643494.955
4,Connaught Place,639756.770
5,Noida Sector 18,639673.995
6,Vaishali,639446.855
7,MG Road Gurgaon,637444.160
8,Noida Sector 62,636248.060
9,Greater Noida,631077.895


### 9.7 Repeat Customer Percentage

In [40]:
repeat = df["Customer_ID"].value_counts()
(repeat > 1).mean() * 100

np.float64(30.70357554786621)

## 10. Final Data Quality Summary
Ensuring dataset is clean and analysis-ready.

In [41]:
print("Final Dataset Shape:", df.shape)
print("Remaining Null Values:\n",([i for i in df.columns if df[i].isnull().sum()>0]))
print("Duplicate Rows:", df.duplicated().sum())

Final Dataset Shape: (45000, 28)
Remaining Null Values:
 []
Duplicate Rows: 0


## 11. Exporting Cleaned Dataset

In [60]:
customers.to_csv("customers_clean.csv", index=False)
drivers.to_csv("drivers_clean.csv", index=False)
vehicles.to_csv("vehicles_clean.csv", index=False)
rides.to_csv("rides_clean.csv", index=False)
ride_status.to_csv("ride_status_clean.csv", index=False)